# 01 Cleaning: Track 2, Cardiovascular Health in Women

This notebook loads each dataset, cleans it and records every rule in `team/docs/cleaning_log.md`.

**How to use it**

- Run the cells from top to bottom (**Runtime > Run all** on Colab).
- Cells marked **You decide** have settings set to `None`. The notebook prints a reminder until you fill them in.
- Only clean the datasets you use. Skip the other sections.

Read [`2-the-data.md`](../../2-the-data.md) first if you have not.

In [ ]:
# Setup: run this cell first. It finds the track folder and the data.
import re, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PUBLIC_REPO = "https://github.com/Harvard-Undergraduate-OpenBio/openbio-datathon-2026.git"
IN_COLAB = "google.colab" in sys.modules

def find_track_dir():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "track-02" / "data").is_dir():
            return folder / "track-02"
        if folder.name == "track-02" and (folder / "data").is_dir():
            return folder
    if IN_COLAB:
        dest = Path("/content/openbio-datathon-2026")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO, str(dest)], check=True)
        return dest / "track-02"
    return None

TRACK = find_track_dir()
TEAM = TRACK / "team" if TRACK else Path.cwd()
CLEAN = TEAM / "data_clean"
FIGURES = TEAM / "figures"
for folder in (CLEAN, FIGURES, TEAM / "docs"):
    folder.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [d for d in [TRACK / "data" if TRACK else None, TRACK / "data" / "gbd" if TRACK else None,
               Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/datathon")] if d]

def missing(name):
    print(f"Fill in {name} in the cell above, then rerun this cell.")

print("Track folder:", TRACK)
print("Team folder: ", TEAM)
if IN_COLAB:
    print("On Colab, outputs are saved in this session only. Download them or copy them into your team repository.")


## Helpers

You do not need to edit this cell. `log_step` records a cleaning rule and how many records it affected.

In [ ]:
LOG = []

def log_step(dataset, rule, before, after, reason=""):
    """Record one cleaning rule. before and after are row counts."""
    LOG.append({"Dataset": dataset, "Rule": rule, "Reason": reason,
                "Records before": before, "Records after": after,
                "Records affected": before - after})
    print(f"[{dataset}] {rule}: {before - after:,} records affected ({after:,} remain)")

def write_log():
    path = TEAM / "docs" / "cleaning_log.md"
    lines = ["# Cleaning Log", "", "Written by `01_cleaning.ipynb`. Rerun the notebook to update it.", "",
             "| Dataset | Rule | Reason | Records before | Records after | Records affected |",
             "|---|---|---|---|---|---|"]
    for r in LOG:
        lines.append(f"| {r['Dataset']} | {r['Rule']} | {r['Reason']} | {r['Records before']:,} | "
                     f"{r['Records after']:,} | {r['Records affected']:,} |")
    path.write_text("\n".join(lines) + "\n")
    print(f"Wrote {len(LOG)} rules to {path}")

def find_file(*names):
    """Return the first non-empty file matching any name. Empty placeholders are skipped."""
    for folder in SEARCH_DIRS:
        for name in names:
            p = folder / name
            if p.is_file() and p.stat().st_size > 0:
                return p
    return None

def load_csv(path, **kwargs):
    """Read a CSV, detecting whether it uses commas or semicolons."""
    with open(path, encoding="utf-8-sig") as f:
        first = f.readline()
    sep = ";" if first.count(";") > first.count(",") else ","
    return pd.read_csv(path, sep=sep, **kwargs)

def load_table(path, header=None):
    """Read a CSV or Excel table. header=None keeps every row so you can see the real headers."""
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path, header=header)
    return load_csv(path, header=header)


---
# Dataset 1: Cardiovascular Disease Dataset

In [ ]:
d1_path = find_file("cardiovascular_dataset_OpenBio_Final.csv")
if d1_path:
    d1_raw = load_csv(d1_path)
    print(f"Dataset 1: {d1_raw.shape[0]:,} rows, {d1_raw.shape[1]} columns")
    display(d1_raw.head())
else:
    d1_raw = None
    print("Dataset 1 not found or still an empty placeholder. Skip this section or add the file to track-02/data/.")


### Step 1: Duplicates and age

Age is stored in days. This converts it to years.

In [ ]:
if d1_raw is not None:
    d1 = d1_raw.copy()
    n = len(d1)
    d1 = d1.drop_duplicates(subset="id")
    log_step("D1", "Drop duplicate ids", n, len(d1), "Each patient should appear once")
    d1["age_years"] = d1["age"] / 365.25
    print(d1["age_years"].describe().round(1))


### Step 2: Check the sex coding

The Kaggle page lists `gender` only as a "categorical code" and does not say which number is which. Women are shorter on average, so the code with the lower mean height is women. In this file that is code 1. If the check below disagrees, change `WOMEN_CODE`.

**You decide:** `D1_SEX_SCOPE`. Use `"women"` to keep only women, or `"both"` to compare women with men.

In [ ]:
WOMEN_CODE = 1
D1_SEX_SCOPE = "women"   # "women" or "both"

if d1_raw is not None:
    print("Mean height (cm) by gender code:")
    print(d1.groupby("gender")["height"].mean().round(1))
    d1["female"] = (d1["gender"] == WOMEN_CODE).astype(int)
    if D1_SEX_SCOPE == "women":
        n = len(d1)
        d1 = d1[d1["female"] == 1].copy()
        log_step("D1", "Keep women only", n, len(d1), "Track focus on women")


### Step 3: Look at the measurements

Before choosing a rule, look at the values. The table shows the smallest and largest values, and the plots show where most values sit.

In [ ]:
MEASURES = {"ap_hi": "Systolic BP (mmHg)", "ap_lo": "Diastolic BP (mmHg)",
            "height": "Height (cm)", "weight": "Weight (kg)"}

if d1_raw is not None:
    display(d1[list(MEASURES)].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).round(1))
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, (col, label) in zip(axes, MEASURES.items()):
        lo, hi = d1[col].quantile([0.005, 0.995])
        d1[col].clip(lo, hi).hist(bins=60, ax=ax)
        ax.set_title(label + "\n(middle 99% shown)")
    plt.tight_layout()
    plt.show()


### Step 4: Plausibility rules

**You decide:** a range for each measurement, written as `(low, high)`, and a one-sentence reason. Rows outside the range are removed. Leave a range as `None` to skip it, but note that impossible values will stay in.

In [ ]:
RANGES = {
    "ap_hi":  {"range": None, "reason": ""},   # e.g. (low, high) in mmHg
    "ap_lo":  {"range": None, "reason": ""},
    "height": {"range": None, "reason": ""},   # cm
    "weight": {"range": None, "reason": ""},   # kg
}

if d1_raw is not None:
    for col, rule in RANGES.items():
        if rule["range"] is None:
            missing(f'RANGES["{col}"]')
            continue
        lo, hi = rule["range"]
        n = len(d1)
        d1 = d1[d1[col].between(lo, hi)]
        log_step("D1", f"Keep {col} between {lo} and {hi}", n, len(d1), rule["reason"])

    n = len(d1)
    d1 = d1[d1["ap_hi"] > d1["ap_lo"]]
    log_step("D1", "Keep rows where systolic > diastolic", n, len(d1), "Systolic is always higher than diastolic")


### Step 5: BMI and ordered categories

BMI is calculated after cleaning height and weight. Cholesterol and glucose become ordered labels so plots and models keep their order.

In [ ]:
LEVELS = {1: "normal", 2: "above normal", 3: "well above normal"}

if d1_raw is not None:
    d1["bmi"] = d1["weight"] / (d1["height"] / 100) ** 2
    for col in ("cholesterol", "gluc"):
        d1[col + "_level"] = pd.Categorical(d1[col].map(LEVELS), categories=list(LEVELS.values()), ordered=True)
    display(d1[["age_years", "bmi", "ap_hi", "ap_lo"]].describe().round(1))
    print(d1["cholesterol_level"].value_counts().sort_index())
    d1.to_csv(CLEAN / "d1_clean.csv", index=False)
    print(f"Saved {len(d1):,} rows to {CLEAN / 'd1_clean.csv'}")


---
# Dataset 2: Heart Attack Risk Prediction Dataset (synthetic)

Column names are converted to lowercase with underscores, so `Heart Attack Risk` becomes `heart_attack_risk`. This makes them easier to use in models.

In [ ]:
d2_path = find_file("heart_attack_prediction_dataset_OpenBio_Final.csv")
if d2_path:
    d2_raw = load_csv(d2_path)
    print(f"Dataset 2: {d2_raw.shape[0]:,} rows, {d2_raw.shape[1]} columns")
    display(d2_raw.head())
else:
    d2_raw = None
    print("Dataset 2 not found or still an empty placeholder. Skip this section or add the file to track-02/data/.")


### Step 1: Rename columns and split blood pressure

`Blood Pressure` is text such as `158/88`. This splits it into two number columns.

In [ ]:
def snake(name):
    return re.sub(r"[^0-9a-z]+", "_", name.strip().lower()).strip("_")

if d2_raw is not None:
    d2 = d2_raw.rename(columns=snake).copy()
    n = len(d2)
    d2 = d2.drop_duplicates(subset="patient_id")
    log_step("D2", "Drop duplicate patient ids", n, len(d2), "Each record should appear once")

    bp = d2["blood_pressure"].astype(str).str.extract(r"^\s*(\d+)\s*/\s*(\d+)\s*$").astype(float)
    d2["bp_systolic"], d2["bp_diastolic"] = bp[0], bp[1]
    n = len(d2)
    d2 = d2.dropna(subset=["bp_systolic", "bp_diastolic"])
    log_step("D2", "Drop rows where blood pressure is not in 'number/number' form", n, len(d2), "Cannot be split")
    display(d2[["blood_pressure", "bp_systolic", "bp_diastolic"]].head())


### Step 2: Encode yes/no and ordered levels

In this file the yes/no columns already hold 1 and 0. That includes Alcohol Consumption: the Kaggle page describes it as levels, but the file stores only 0 or 1. Diet is text and keeps its order: Unhealthy, Average, Healthy. Sex becomes a `female` column (1 or 0). Any value the notebook does not recognize is printed so you can decide what to do.

In [ ]:
YES_NO = {"yes": 1, "no": 0, "1": 1, "0": 0, "true": 1, "false": 0}
BINARY = ["diabetes", "family_history", "smoking", "obesity", "previous_heart_problems",
          "medication_use", "heart_attack_risk"]
ALCOHOL_ORDER = ["None", "Light", "Moderate", "Heavy"]
DIET_ORDER = ["Unhealthy", "Average", "Healthy"]

if d2_raw is not None:
    for col in BINARY:
        if col not in d2:
            continue
        mapped = d2[col].astype(str).str.strip().str.lower().map(YES_NO)
        bad = d2.loc[mapped.isna(), col].unique()
        if len(bad):
            print(f"{col}: unrecognized values {list(bad)[:5]}")
        d2[col] = mapped

    alc = d2["alcohol_consumption"].astype(str).str.strip()
    if alc.str.title().isin(ALCOHOL_ORDER).all():
        d2["alcohol_level"] = pd.Categorical(alc.str.title(), categories=ALCOHOL_ORDER, ordered=True)
        d2["alcohol_code"] = d2["alcohol_level"].cat.codes
        print("Alcohol is stored as levels:", ALCOHOL_ORDER)
    else:
        d2["alcohol_consumption"] = alc.str.lower().map(YES_NO)
        print("Alcohol is stored as yes/no, encoded as 1/0")

    d2["diet_level"] = pd.Categorical(d2["diet"].str.strip().str.title(), categories=DIET_ORDER, ordered=True)
    d2["diet_code"] = d2["diet_level"].cat.codes
    d2["female"] = (d2["sex"].str.strip().str.lower() == "female").astype(int)
    print(d2["diet_level"].value_counts().sort_index())


### Step 3: Choose who to analyze

**You decide:** `SEX_SCOPE`. Use `"women"` to keep only women, or `"both"` to compare women with men (use the `female` column as a predictor). Write the reason.

In [ ]:
SEX_SCOPE = None        # "women" or "both"
SEX_SCOPE_REASON = ""

if d2_raw is not None:
    print(d2["sex"].value_counts())
    if SEX_SCOPE is None:
        missing("SEX_SCOPE")
        print("Saving both sexes for now.")
    elif SEX_SCOPE == "women":
        n = len(d2)
        d2 = d2[d2["female"] == 1]
        log_step("D2", "Keep women only", n, len(d2), SEX_SCOPE_REASON)
    d2.to_csv(CLEAN / "d2_clean.csv", index=False)
    print(f"Saved {len(d2):,} rows to {CLEAN / 'd2_clean.csv'}")


---
# Dataset 3: Global CVD in Women (Tables S1 and S2)

These tables are copied cell for cell from the article's supplement, so they still look like printed tables:

- The first row of each file is the table title.
- Table S2 spreads its header over two rows, with `1990` and `2019` sitting above the column names, and has empty spacer columns.
- Table S1 has label rows with no numbers (`Socio-demographic index`, `GBD regions`) and a footnote in the last row.
- Estimates and their intervals share one cell, such as `1083.58 (1011.93, 1164.76)`.

The steps are:

1. Look at the raw rows and find the real header row or rows.
2. Split intervals stored as text into three numbers.
3. Standardize country names.
4. Reshape from wide to long if you need one row per year.

In [ ]:
s1_path = find_file("gbd/table_s1.xlsx", "gbd/table_s1.csv", "table_s1.xlsx", "table_s1.csv")
s2_path = find_file("gbd/table_s2.xlsx", "gbd/table_s2.csv", "table_s2.xlsx", "table_s2.csv")
s1_raw = load_table(s1_path) if s1_path else None
s2_raw = load_table(s2_path) if s2_path else None
for name, raw in (("S1", s1_raw), ("S2", s2_raw)):
    if raw is None:
        print(f"Table {name} not found. Add it to track-02/data/gbd/ as table_{name.lower()}.xlsx or .csv")
    else:
        print(f"Table {name}: {raw.shape[0]} raw rows, {raw.shape[1]} raw columns. First 8 and last 2 rows (row numbers on the left):")
        display(pd.concat([raw.head(8), raw.tail(2)]))

### Step 1: Set the header row

**You decide:** the row number (left column above) that holds the real column names. Rows above it are dropped. If the header is spread over two rows, give both in a list such as `[1, 2]` and they are joined into one name per column.

What the cell does once you fill it in:

- Names become lowercase with underscores, so `MIR per 100,000 in 1990` becomes `mir_per_100_000_in_1990`. With a two-row header the lower row comes first, so `1990` above `EAPC (95% CI)` becomes `eapc_95_ci_1990`.
- Empty spacer columns are dropped. A column with no header is named by its position, such as `column_0`.
- Spaces around values are trimmed, and columns that hold only plain numbers become numbers.
- The column names are printed. Use `S1_RENAME` or `S2_RENAME` to shorten any of them.

**You decide (Table S1):** `S1_LABEL_ROWS`. Rows that have a label but no numbers are section headings or footnotes. Use `"section"` to move each label into a new `section` column for the rows under it, or `"keep"` to leave the rows as they are.

In [ ]:
S1_HEADER_ROW = None   # a row number, or a list such as [1, 2] for a two-row header
S2_HEADER_ROW = None
S1_RENAME = None       # optional, e.g. {"column_0": "region"}
S2_RENAME = None       # optional, e.g. {"long_printed_name": "short_name"}
S1_LABEL_ROWS = None   # "section" or "keep"
S2_LABEL_ROWS = None   # only needed if the cell reports label rows in Table S2

def tidy_name(text):
    return re.sub(r"[^0-9a-z]+", "_", str(text).strip().lower()).strip("_")

def apply_header(raw, rows, name, rename=None):
    if raw is None:
        return None
    if rows is None:
        missing(f"{name}_HEADER_ROW")
        return None
    rows = [rows] if isinstance(rows, int) else list(rows)
    table = raw.iloc[max(rows) + 1:]
    keep = table.notna().any()                        # drop empty spacer columns
    table = table.loc[:, keep].copy()
    header = raw.loc[rows, keep].astype(object)
    upper = header.iloc[:-1].ffill(axis=1)            # a merged header cell covers the columns to its right
    names = []
    for i, col in enumerate(header.columns):
        parts = [header.iloc[-1][col]] + [upper.loc[r, col] for r in reversed(upper.index)]
        parts = [tidy_name(p) for p in parts if pd.notna(p)]
        names.append("_".join(p for p in dict.fromkeys(parts) if p) or f"column_{i}")
    if len(set(names)) < len(names):
        print(f"{name}: some column names repeat. The header may span two rows: try a list such as [{rows[0] - 1}, {rows[0]}].")
        seen = {}
        for i, n in enumerate(names):
            seen[n] = seen.get(n, 0) + 1
            if seen[n] > 1:
                names[i] = f"{n}_{seen[n]}"
    table.columns = names
    table = table.dropna(how="all").reset_index(drop=True)
    for col in table.columns:
        table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
        numbers = pd.to_numeric(table[col], errors="coerce")
        if numbers.notna().sum() == table[col].notna().sum() and numbers.notna().any():
            table[col] = numbers
    if rename:
        table = table.rename(columns=rename)
    print(f"{name}: {len(table)} rows. Columns: {list(table.columns)}")
    return table

def handle_label_rows(table, choice, name):
    """Rows with a label in the first column and nothing else are section headings or footnotes."""
    if table is None:
        return None
    is_label = table.iloc[:, 0].notna() & table.iloc[:, 1:].isna().all(axis=1)
    if not is_label.any():
        return table
    print(f"{name}: {int(is_label.sum())} rows have a label but no values: {table.loc[is_label].iloc[:, 0].tolist()}")
    if choice is None:
        missing(f"{name}_LABEL_ROWS")
    elif choice == "section":
        n = len(table)
        table["section"] = table.iloc[:, 0].where(is_label).ffill()
        table = table[~is_label].reset_index(drop=True)
        log_step(name, "Move label rows into a section column", n, len(table),
                 "Section headings and footnotes are not data rows")
    return table

s1 = handle_label_rows(apply_header(s1_raw, S1_HEADER_ROW, "S1", S1_RENAME), S1_LABEL_ROWS, "S1")
s2 = handle_label_rows(apply_header(s2_raw, S2_HEADER_ROW, "S2", S2_RENAME), S2_LABEL_ROWS, "S2")

### Step 2: Split confidence intervals

Values such as `1083.58 (1011.93, 1164.76)` become three columns: `_est`, `_low` and `_high`. The function handles commas, "to", dashes and the minus sign.

**You decide:** which columns hold values with intervals. Use the column names printed in Step 1. Table S1 has no intervals, so its list starts empty.

In [ ]:
def split_ci(text):
    """'1083.58 (1011.93, 1164.76)' -> (1083.58, 1011.93, 1164.76)."""
    if pd.isna(text):
        return (np.nan, np.nan, np.nan)
    s = str(text).replace("\u2212", "-")                  # unicode minus
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)            # thousands separators
    nums = [float(x) for x in re.findall(r"(?<![\d.])-?\d+(?:\.\d+)?", s)]
    if len(nums) >= 3:
        return tuple(nums[:3])
    if len(nums) == 1:
        return (nums[0], np.nan, np.nan)
    return (np.nan, np.nan, np.nan)

# Quick check
for example in ["1083.58 (1011.93, 1164.76)", "-0.25 (-0.31, -0.2)", "963.95 (850.1 to 1080.2)",
                "-0.52 (\u22120.61, \u22120.43)", "1,234,567 (1,100,000-1,300,000)"]:
    print(example, "->", split_ci(example))

S1_CI_COLUMNS = []     # Table S1 has no intervals
S2_CI_COLUMNS = None   # e.g. ["asir_per_100_000_no_95_ui_2019"]

def split_columns(table, cols, name):
    if table is None:
        return
    if cols is None:
        missing(f"{name}_CI_COLUMNS")
        return
    for col in cols:
        if col not in table.columns:
            print(f"{name}: no column called {col!r}. Columns are {list(table.columns)}")
            continue
        parts = table[col].apply(split_ci)
        table[col + "_est"] = [p[0] for p in parts]
        table[col + "_low"] = [p[1] for p in parts]
        table[col + "_high"] = [p[2] for p in parts]
        print(f"{name} {col}: {table[col + '_est'].isna().sum()} values could not be read")

split_columns(s1, S1_CI_COLUMNS, "S1")
split_columns(s2, S2_CI_COLUMNS, "S2")

### Step 3: Standardize country names

Different sources spell countries differently. Table S2 uses the long GBD names, such as `United States of America` and `Viet Nam`, while Dataset 2 uses `United States` and `Vietnam`. This trims spaces, removes footnote marks and maps common long names to short ones. Add to `COUNTRY_FIXES` if you combine sources.

**You decide:** the column that holds country names. In Table S2 its printed header is `Region`.

In [ ]:
COUNTRY_FIXES = {
    "United States of America": "United States",
    "Russian Federation": "Russia",
    "Viet Nam": "Vietnam",
    "Iran (Islamic Republic of)": "Iran",
    "Republic of Korea": "South Korea",
    "Democratic People's Republic of Korea": "North Korea",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "United Republic of Tanzania": "Tanzania",
    "Lao People's Democratic Republic": "Laos",
    "Syrian Arab Republic": "Syria",
    "Republic of Moldova": "Moldova",
    "Micronesia (Federated States of)": "Micronesia",
    "Taiwan (Province of China)": "Taiwan",
    "T\u00fcrkiye": "Turkey",
    "C\u00f4te d'Ivoire": "Cote d'Ivoire",
}

def clean_country(name):
    if pd.isna(name):
        return name
    s = re.sub(r"\s+", " ", str(name)).strip()
    s = re.sub(r"[\*\u2020\u2021]+$|\s*\[\d+\]$", "", s).strip()   # footnote marks
    return COUNTRY_FIXES.get(s, s)

S2_COUNTRY_COLUMN = None   # e.g. "region"

if s2 is not None:
    if S2_COUNTRY_COLUMN is None:
        missing("S2_COUNTRY_COLUMN")
    else:
        before = s2[S2_COUNTRY_COLUMN].nunique()
        s2[S2_COUNTRY_COLUMN] = s2[S2_COUNTRY_COLUMN].map(clean_country)
        print(f"{before} names before, {s2[S2_COUNTRY_COLUMN].nunique()} after")

### Step 4: Reshape wide to long (optional)

Wide tables have one column per year, such as an ASIR column for 1990 and another for 2019. Long tables have one row per country and year, which is easier to plot.

**You decide:** the columns that identify a row (`ID_COLUMNS`) and the columns to stack (`VALUE_COLUMNS`). The year is read from each column name, so every stacked column needs a year in its name.

In [ ]:
S2_ID_COLUMNS = None      # e.g. ["region", "sdi_value_in_2019"]
S2_VALUE_COLUMNS = None   # e.g. ["asir_per_100_000_no_95_ui_1990_est", "asir_per_100_000_no_95_ui_2019_est"]

s2_long = None
if s2 is not None:
    if S2_ID_COLUMNS is None or S2_VALUE_COLUMNS is None:
        missing("S2_ID_COLUMNS and S2_VALUE_COLUMNS")
    else:
        s2_long = s2.melt(id_vars=S2_ID_COLUMNS, value_vars=S2_VALUE_COLUMNS,
                          var_name="measure", value_name="value")
        s2_long["year"] = s2_long["measure"].str.extract(r"((?:19|20)\d{2})")[0].astype(int)
        s2_long["measure"] = s2_long["measure"].str.replace(r"_?(?:19|20)\d{2}", "", regex=True).str.strip("_")
        display(s2_long.head())

for name, table in (("s1", s1), ("s2", s2), ("s2_long", s2_long)):
    if table is not None:
        table.to_csv(CLEAN / f"{name}_clean.csv", index=False)
        print(f"Saved {name}_clean.csv ({len(table)} rows)")

---
# Write the Cleaning Log

Run this last. Open `team/docs/cleaning_log.md` afterwards and check that each rule has a reason.

In [ ]:
write_log()
display(pd.DataFrame(LOG))
